In [13]:
import xml
import xmltodict

In [19]:
class AudioRecording():
    def __init__(self,filename,startTime):
        print(filename)
        self.filename = filename.split('/')[1] + '.flv'
        self.new_filename = None
        self.delay = int(startTime)

class VideoRecording():
    def __init__(self,filename,startTime):
        self.filename = filename.split('/')[1] + '.flv'
        self.new_filename = None
        self.delay = int(startTime)

In [3]:
#opening xml file
with open(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\plain_course_with_video\indexstream.xml') as fd:
    index = xmltodict.parse(fd.read())



In [4]:
type(index)


collections.OrderedDict

In [5]:
for item in index['root']['Message']:
    if item.get('String') and item.get('String') == 'streamAdded':
        print(item['Array']['Object']['startTime'],':',item['Array']['Object']['streamName'])


131 : /screenshare_4_2
156 : /cameraVoip_3_4


In [24]:
import pathlib
def parseindexstream(path):
    with open(str(pathlib.Path(path))) as fd:
        index = xmltodict.parse(fd.read())
    audio = []
    for item in index['root']['Message']:
        if item.get('String') and item.get('String') == 'streamAdded':
            if 'cameravoip' in str(item['Array']['Object']['streamName']).lower():
                audio.append(AudioRecording(item['Array']['Object']['streamName'],item['Array']['Object']['startTime']))
    sorted_audio = sorted(audio, key=lambda audio: audio.delay)
    for i, item in enumerate(sorted_audio):
        sorted_audio[i].new_filename = f"audio_{i}"

    video = []
    for item in index['root']['Message']:
        if item.get('String') and item.get('String') == 'streamAdded':
            if 'screenshare' in str(item['Array']['Object']['streamName']).lower():
                video.append(VideoRecording(item['Array']['Object']['streamName'],item['Array']['Object']['startTime']))
    sorted_video = sorted(video, key=lambda video: video.delay)
    for i, item in enumerate(sorted_video):
        sorted_video[i].new_filename = f"video_{i}"

    return {'audio':sorted_audio,'video':sorted_video}

In [25]:
result_plain_course_with_video = parseindexstream(pathlib.Path(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\plain_course_with_video\indexstream.xml'))

/cameraVoip_3_4


In [26]:

a1 = parseindexstream(pathlib.Path(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\multiple_audio_video_course\indexstream.xml'))
a2 = parseindexstream(pathlib.Path(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\plain_course_with_video\indexstream.xml'))
a3 = parseindexstream(pathlib.Path(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\only_audio_course\indexstream.xml'))
a4 = parseindexstream(pathlib.Path(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\uploaded_mp4_course\indexstream.xml'))
a5 = parseindexstream(pathlib.Path(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\pptx_asset_audio_course\indexstream.xml'))


/cameraVoip_0_3
/cameraVoip_0_6
/cameraVoip_3_7
/cameraVoip_3_4
/cameraVoip_0_3
/cameraVoip_2_4
/cameraVoip_3_5
/cameraVoip_4_6
/cameraVoip_1_2
/cameraVoip_0_3
/cameraVoip_0_4
/cameraVoip_1_5
/cameraVoip_1_6


Main stream parse


In [2]:
def extract_directory(url):
    url = url.split('?download-url=')[1]
    url = url.lstrip()
    url = url.rstrip()
    url = '/'.join(url.split('/')[2:] if url.split('/')[0] == '' else url.split('/')[1:])
    url = ''.join(url.split('&name='))
    return '/'+url

def extract_filename(url):
    query = str(url.split('/')[-1]).split('&')
    for par in query:
        if 'name=' in par:
            return par.split('name=')[1]

def extract_filetype(filename):
        return filename.split('.')[-1]

class assetDocument():
    def __init__(self,assetURL,special=False):
        self.filename= extract_filename(url=assetURL) if not special else assetURL.split('/')[-1]
        self.directory = extract_directory(url=assetURL) if not special else assetURL
        self.filetype= extract_filetype(filename=self.filename)

    def __eq__(self, other):
        return self.filename == other.filename
    def __hash__(self):
        return hash(self.filename)


In [3]:

#opening xml file
with open(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\pptx_asset_audio_course\mainstream.xml') as fd:
    index = xmltodict.parse(fd.read())



In [4]:
type(index)


collections.OrderedDict

In [5]:
files = []
for item in index['root']['Message']:
    if item.get('String') and item.get('String') == 'setContentSo':
        try:
            urladd = item['Array']['Object']['newValue']['documentDescriptor']['downloadUrl']
            assert urladd,'empty urladd'
        except:
            pass
        else:
            files.append(
                assetDocument(assetURL=urladd)
            )
        try:
            urladd = item['Array']['Object']['newValue']['documentDescriptor']['playbackFileNameHTMLClient']
            thename = item['Array']['Object']['newValue']['documentDescriptor']['theName']
            assert all([thename,urladd])
        except:
            continue
        else:
            print(urladd+'default/'+thename)
            files.append(
                assetDocument(assetURL=urladd+'default/'+thename,special=True)
            )

print(files)

/_a7/p2qczwk9tmwe/default/viewer.swfdefault/tmp.ppt


In [11]:
import pathlib
def parsemainstream(path):
    with open(str(pathlib.Path(path))) as fd:
        index = xmltodict.parse(fd.read())
    assets = []
    for item in index['root']['Message']:
        if item.get('String') and item.get('String') == 'setContentSo':
            try:
                urladd = item['Array']['Object']['newValue']['documentDescriptor']['downloadUrl']
                assert urladd,'empty urladd document'
            except:
                pass
            else:
                assets.append(
                    assetDocument(assetURL=urladd)
                )
                continue
            try:
                urladd = item['Array']['Object']['newValue']['documentDescriptor']['playbackFileNameHTMLClient']
                thename = item['Array']['Object']['newValue']['documentDescriptor']['theName']
                assert all([thename,urladd])
            except:
                continue
            else:
                if 'mp4' not in thename.split('.')[-1]:
                    continue
                assets.append(
                    assetDocument(assetURL=urladd+'default/'+thename,special=True)
                )

    return list(set(assets)) if assets else None


In [12]:
m1 = parsemainstream(pathlib.Path(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\multiple_audio_video_course\mainstream.xml'))
m2 = parsemainstream(pathlib.Path(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\plain_course_with_video\mainstream.xml'))
m3 = parsemainstream(pathlib.Path(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\only_audio_course\mainstream.xml'))
m4 = parsemainstream(pathlib.Path(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\uploaded_mp4_course\mainstream.xml'))
m5 = parsemainstream(pathlib.Path(r'C:\Users\Soroush\PycharmProjects\acdii\test_adobeconnects\pptx_asset_audio_course\mainstream.xml'))

In [17]:
import logging,settings
logging.basicConfig(filename=settings.SETTINGS['LOG_PATH'],filemode='a',level=logging.DEBUG,datefmt='%d-%b-%y %H:%M:%S',format="%(levelname)s -- %(asctime)s -- %(name)s -- %(message)s")
logging.debug('test')
